# 05b — Parallel Tempering (Replica Exchange NUTS) — DATA

**Hypothesis H3**: Parallel tempering with a geometric temperature ladder
overcomes the singular geometry that defeats fixed mass-matrix NUTS.
Hot chains (β < 1) flatten the posterior, making degenerate directions
easier to traverse. Replica-exchange swaps propagate this exploration to
the cold chain (β = 1) which targets the true posterior.

**Hypothesis H3-1 (iterative refinement, mirrors H1-1)**: Starting from
H3's adapted per-chain mass matrices, repeated rounds of
(warmup → production → use new per-chain M's) drive the cold-chain
mixing metrics (ACF@50, ESS, sample-quality variance) to a stationary
regime. This lets us directly compare PT's convergence trajectory
against H1-1 (single-chain NUTS without Hessian init).

**This notebook produces sampling artifacts.** Analysis lives in
`05b_sampling_tempered_analysis.ipynb`.

**Temperature ladder**: β = [1.0, 0.85, 0.7, 0.55, 0.45, 0.35, 0.25, 0.15]
— denser spacing with 8 rungs to maintain swap connectivity.

**Outputs** (per `RUN_NAME`):
- `data/results_tempered/{RUN_NAME}/nuts_samples.pt` — H3 initial PT run
- `data/results_tempered/{RUN_NAME}/iterative/round_NN.pt` — H3-1 per-round artifacts
- `data/results_tempered/{RUN_NAME}/iterative/round_summary_raw.json` — H3-1 loop summary

**Success criteria**:
- H3: cold-chain ACF@50 median drops below 0.3 (vs ~0.97 in H1)
- H3-1: ACF@50 / ESS stabilize across rounds (convergence comparable to H1-1)


In [1]:
import collections
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# Resolve project root
cwd = Path.cwd().resolve()
project_root = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "packages" / "pytorch_models" / "markov_transformer.py").exists()
    ),
    None,
)
if project_root is None:
    alt_root = cwd / "projects" / "markov-chain-learning"
    if (alt_root / "packages" / "pytorch_models" / "markov_transformer.py").exists():
        project_root = alt_root
if project_root is None:
    raise RuntimeError("Could not locate markov-chain-learning project root")

packages_dir = project_root / "packages"
if str(packages_dir) not in sys.path:
    sys.path.insert(0, str(packages_dir))

from pytorch_models import MarkovTransformer

DATA_DIR = project_root / "experiments" / "single-chain" / "data"

print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")

Project root: /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning
Data dir: /Users/ashrafahmed/code/slt-deep/projects/markov-chain-learning/experiments/single-chain/data


In [2]:
# ── Parameters (papermill) ──────────────────────────────────────────────────
# This cell is tagged "parameters" for papermill injection.

# Run identification
RUN_NAME: str = "default"

# Temperature ladder (H3)
BETAS: list = [
    1.0,
    0.85,
    0.7,
    0.55,
    0.45,
    0.35,
    0.25,
    0.15,
]
SWAP_EVERY: int = 10

# H3: initial PT run (each chain adapts its own M from identity)
N_WARMUP: int = 250
N_SAMPLES: int = 50
MAX_TREE_DEPTH: int = 7
TARGET_ACCEPT: float = 0.69

# Prior
SIGMA_PRIOR: float = 10.0

# Initialisation
INIT_PERTURB_SCALE: float = 1.0

# H3-1: iterative refinement (mirrors H1-1)
RUN_ITERATIVE: bool = True
N_ROUNDS: int = 5
N_WARMUP_PER_ROUND: int = 250
N_SAMPLES_PER_ROUND: int = 50
RESUME: bool = False  # resume from highest existing round_NN.pt


In [3]:
# Parameters
BETAS = [1.0, 0.85, 0.7, 0.55, 0.45, 0.35, 0.25, 0.15]
RUN_NAME = "default"
SWAP_EVERY = 1
N_WARMUP = 2000
N_SAMPLES = 500
MAX_TREE_DEPTH = 4
TARGET_ACCEPT = 0.6
SIGMA_PRIOR = 10.0
INIT_PERTURB_SCALE = 1.0
RUN_ITERATIVE = "true"
N_ROUNDS = 5
N_WARMUP_PER_ROUND = 2000
N_SAMPLES_PER_ROUND = 500
RESUME = "false"


In [4]:
# Load dataset
data = torch.load(DATA_DIR / "sequences.pt", weights_only=False)
sequences = data["sequences"]
data_cfg = data["config"]

VOCAB_SIZE = int(data_cfg["n_states"])
MAX_LEN = int(data_cfg["L"])
PAD_ID = int(data_cfg.get("pad_id", -1))
DGP_REGIME = data_cfg.get("dgp_regime", "unknown")

print(f"DGP regime: {DGP_REGIME}")
print(
    f"Sequences: {tuple(sequences.shape)}, VOCAB_SIZE={VOCAB_SIZE}, MAX_LEN={MAX_LEN}"
)

# Load trained model from checkpoint
device = torch.device("cpu")
D_MODEL = VOCAB_SIZE * 2

model = MarkovTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
).to(device)

ckpt = torch.load(DATA_DIR / "checkpoint_single_chain.pt", weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Loaded checkpoint (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")
print(f"Total parameters: {total_params:,}")

# Prepare data tensors
x_data = sequences[:, :-1].to(device)
y_data = sequences[:, 1:].to(device)
x_data = x_data.clone()
x_data[x_data == PAD_ID] = 0

mle_param = torch.cat([p.flatten() for p in model.parameters()]).detach()
print(f"MLE parameter vector: d = {mle_param.shape[0]}")

DGP regime: single
Sequences: (3000, 10), VOCAB_SIZE=5, MAX_LEN=10
Loaded checkpoint (epoch 73, val_loss=1.2806)
Total parameters: 910
MLE parameter vector: d = 910


## Define Log-Likelihood and Prior

In [5]:
from torch_bdn.bn import BayesianNet
from torch_bdn.sampling import NUTS, Perturb, Sampler


def loss_fn(logits, targets):
    ce = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
        reduction="none",
        ignore_index=PAD_ID,
    )
    ce = ce.view(logits.shape[:-1])
    mask = (targets != PAD_ID).float()
    return (ce * mask).sum() / mask.sum()


def make_prior_logp(mu: torch.Tensor, sigma=10.0):
    """Gaussian prior N(mu, sigma^2 I) — centred at the MAP."""
    mean = mu.detach().clone()

    def prior_logp(params):
        flat = torch.cat([p.flatten() for p in params])
        diff = flat - mean
        return -0.5 * diff.pow(2).sum() / (sigma**2)

    return prior_logp


bn = BayesianNet(
    model, loss_fn, make_prior_logp(mle_param, sigma=SIGMA_PRIOR), compile=True
)
print(f"BayesianNet ready: d={mle_param.shape[0]}, prior σ={SIGMA_PRIOR}")

BayesianNet ready: d=910, prior σ=10.0


## H3: Initial Parallel Tempering Run

**Temperature ladder design**: Denser spacing with 8 rungs to maintain
swap connectivity (previous run showed 0% acceptance at large gaps).
- β = 1.0  — cold chain (true posterior)
- β = 0.85 — slight tempering
- β = 0.7  — mild tempering
- β = 0.55 — moderate tempering
- β = 0.45 — moderate-strong tempering
- β = 0.35 — strong tempering
- β = 0.25 — hot
- β = 0.15 — very hot (nearly flat posterior)

**Physics**: At inverse temperature β, the posterior becomes
π_β(θ) ∝ p(D|θ)^β · p(θ). The log-density curvature scales as β·H,
so degenerate directions (H≈0) remain flat while non-degenerate
directions become shallower.

**Swap frequency**: Every 10 NUTS samples, propose DEO adjacent swaps.


In [6]:
# ── Tempering configuration (derived from parameters) ──
N_CHAINS = len(BETAS)

print(f"Run: {RUN_NAME}")
print(f"Temperature ladder: β = {BETAS}")
print(f"  {N_CHAINS} chains, swap every {SWAP_EVERY} samples")
print(f"  Warmup: {N_WARMUP}, Production: {N_SAMPLES}")
print(f"  Tree depth: {MAX_TREE_DEPTH}, target accept: {TARGET_ACCEPT}")
print(f"  Prior σ: {SIGMA_PRIOR}, init perturb: {INIT_PERTURB_SCALE}")
print("\n  Effective curvature scaling at each β:")
for beta in BETAS:
    print(f"    β={beta:.1f} → H_eff = {beta:.1f}·H  (ridge height × {beta:.1f})")

Run: default
Temperature ladder: β = [1.0, 0.85, 0.7, 0.55, 0.45, 0.35, 0.25, 0.15]
  8 chains, swap every 1 samples
  Warmup: 2000, Production: 500
  Tree depth: 4, target accept: 0.6
  Prior σ: 10.0, init perturb: 1.0

  Effective curvature scaling at each β:
    β=1.0 → H_eff = 1.0·H  (ridge height × 1.0)
    β=0.8 → H_eff = 0.8·H  (ridge height × 0.8)
    β=0.7 → H_eff = 0.7·H  (ridge height × 0.7)
    β=0.6 → H_eff = 0.6·H  (ridge height × 0.6)
    β=0.5 → H_eff = 0.5·H  (ridge height × 0.5)
    β=0.3 → H_eff = 0.3·H  (ridge height × 0.3)
    β=0.2 → H_eff = 0.2·H  (ridge height × 0.2)
    β=0.1 → H_eff = 0.1·H  (ridge height × 0.1)


## Configure and Run Tempered NUTS

Using `torch_bdn`'s built-in parallel tempering: `sampler.sample(...)` with
`betas=[...]` and `swap_every=K`. The API handles:
- Independent NUTS per chain at each temperature
- DEO (Deterministic Even-Odd) swap proposals between adjacent rungs
- Replica-exchange Metropolis-Hastings acceptance criterion
- Tracking swap acceptance counts

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# PARALLEL TEMPERING SAMPLING
# ══════════════════════════════════════════════════════════════════════════════
import time

sampler = Sampler(bn, x_data, y_data)

print(f"Starting parallel tempering: {N_CHAINS} chains × {N_SAMPLES} samples")
print(f"  β = {BETAS}, swap_every = {SWAP_EVERY}")
print(f"  This will take a while (d={mle_param.shape[0]}, depth={MAX_TREE_DEPTH})...")

t0 = time.time()

result = sampler.sample(
    config=NUTS(
        n_warmup=N_WARMUP,
        step_size=0.01,
        max_tree_depth=MAX_TREE_DEPTH,
        target_accept=TARGET_ACCEPT,
        adapt_mass_matrix=True,
    ),
    n_samples=N_SAMPLES,
    n_chains=N_CHAINS,
    init_strategy=Perturb(scale=INIT_PERTURB_SCALE),
    swap_every=SWAP_EVERY,
    betas=BETAS,
)

elapsed = time.time() - t0
print(f"\n✓ Sampling complete in {elapsed / 60:.1f} min")

# ── Swap evidence from swap_history ──
print("\n  Swap log:")
print(f"    Total proposals: {result.n_swaps_proposed}")
print(f"    Accepted: {result.n_swaps_accepted}")
print(f"    Acceptance rate: {result.swap_acceptance_rate():.3f}")

# Per-pair breakdown from swap_history
if result.swap_history:
    from collections import Counter

    pair_proposed = Counter()
    pair_accepted = Counter()
    for entry in result.swap_history:
        pair = entry["pair"]
        pair_proposed[pair] += 1
        if entry["accepted"]:
            pair_accepted[pair] += 1

    print("\n    Per-pair swap rates:")
    for pair in sorted(pair_proposed.keys()):
        i, j = pair
        n_prop = pair_proposed[pair]
        n_acc = pair_accepted[pair]
        rate = n_acc / n_prop if n_prop > 0 else 0
        print(
            f"      β={BETAS[i]:.1f} ↔ β={BETAS[j]:.1f}: {n_acc}/{n_prop} = {rate:.3f}"
        )

    # Show first few swap events as evidence
    accepted_swaps = [e for e in result.swap_history if e["accepted"]]
    print(f"\n    First 10 accepted swaps (of {len(accepted_swaps)} total):")
    for e in accepted_swaps[:10]:
        i, j = e["pair"]
        print(
            f"      round {e['round']:>4d}: β={BETAS[i]:.1f} ↔ β={BETAS[j]:.1f}  (log α = {e['log_alpha']:.2f})"
        )
else:
    print("    ⚠ No swap_history available")

# Per-chain summary (β read from diagnostics)
print("\n  Per-chain summary:")
for ci, ch in enumerate(result.chains):
    diag = ch.diagnostics
    beta_reported = diag.get("beta", None)
    beta_str = (
        f"β={beta_reported:.2f}" if beta_reported is not None else "β=? (not in diag)"
    )
    eps = diag.get("adapted_step_size", diag.get("step_size", "?"))
    print(
        f"    Chain {ci} ({beta_str}): "
        f"accept={ch.acceptance_rate:.3f}, "
        f"ε={eps:.4e}, "
        f"mean_depth={diag.get('mean_tree_depth', 0):.1f}"
    )

Starting parallel tempering: 8 chains × 500 samples
  β = [1.0, 0.85, 0.7, 0.55, 0.45, 0.35, 0.25, 0.15], swap_every = 1
  This will take a while (d=910, depth=4)...
  [PT] Initialising 8 chain replicas...
  [PT] ═══ WARMUP (2000 steps × 8 chains) ═══


  [NUTS warmup c0|β=1.00] step 10/2001  ε=1.56e-02  depth=4 (hit max)  L=15  α=0.50  divs=3/10  mass=identity


  [NUTS warmup c0|β=1.00] step 20/2001  ε=1.17e-02  depth=4 (hit max)  L=15  α=0.67  divs=0/10  mass=identity


  [NUTS warmup c0|β=1.00] step 30/2001  ε=1.18e-02  depth=4 (hit max)  L=15  α=0.59  divs=0/10  mass=identity


  [NUTS warmup c0|β=1.00] step 40/2001  ε=1.63e-02  depth=4 (hit max)  L=15  α=0.61  divs=0/10  mass=identity


...

  [NUTS warmup c0|β=1.00] step 460/2001  ε=1.01e-02  depth=4 (hit max)  L=15  α=0.51  divs=1/10  mass=full


  [NUTS warmup c0|β=1.00] step 470/2001  ε=2.01e-03  depth=4 (hit max)  L=15  α=0.62  divs=0/10  mass=full


  [NUTS warmup c0|β=1.00] step 480/2001  ε=1.57e-03  depth=4 (hit max)  L=15  α=0.58  divs=0/10  mass=full


  [NUTS warmup c0|β=1.00] step 490/2001  ε=3.88e-03  depth=4 (hit max)  L=15  α=0.61  divs=0/10  mass=full


  [NUTS warmup c0|β=1.00] step 500/2001  ε=1.13e-02  depth=4 (hit max)  L=15  α=0.56  divs=0/10  mass=full


In [ ]:
# ── Persist initial-run tempered samples + per-chain adapted M / ε ──
import json
import time as _time

RESULTS_DIR = DATA_DIR / "results_tempered" / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ITER_DIR = RESULTS_DIR / "iterative"

tempered_path = RESULTS_DIR / "nuts_samples.pt"

# Extract per-chain adapted M and step size (used as H3-1 seed)
per_chain_M = []
per_chain_eps = []
for ci, ch in enumerate(result.chains):
    diag = ch.diagnostics
    M = diag.get("adapted_mass_matrix")
    eps = diag.get("adapted_step_size", diag.get("step_size", 0.01))
    per_chain_M.append(M.cpu() if M is not None else None)
    per_chain_eps.append(float(eps))

chains_payload = [
    {
        "parameters": torch.stack(ch.parameters).cpu(),
        "log_probabilities": torch.tensor(ch.log_probabilities).cpu()
        if ch.log_probabilities
        else None,
        "acceptance_rate": ch.acceptance_rate,
        "diagnostics": ch.diagnostics,
        "beta": BETAS[ci],
    }
    for ci, ch in enumerate(result.chains)
]

# Full parameter snapshot for reproducibility
run_params = {
    "run_name": RUN_NAME,
    "betas": BETAS,
    "n_chains": N_CHAINS,
    "swap_every": SWAP_EVERY,
    "n_warmup": N_WARMUP,
    "n_samples": N_SAMPLES,
    "max_tree_depth": MAX_TREE_DEPTH,
    "target_accept": TARGET_ACCEPT,
    "sigma_prior": SIGMA_PRIOR,
    "init_perturb_scale": INIT_PERTURB_SCALE,
    "dgp_regime": DGP_REGIME,
    "d": int(mle_param.shape[0]),
    "elapsed_seconds": elapsed,
    "timestamp": _time.strftime("%Y-%m-%dT%H:%M:%S"),
    # H3-1 parameters (recorded here for the analysis notebook)
    "run_iterative": RUN_ITERATIVE,
    "n_rounds": N_ROUNDS,
    "n_warmup_per_round": N_WARMUP_PER_ROUND,
    "n_samples_per_round": N_SAMPLES_PER_ROUND,
}

# Compute per-pair swap rates from swap_history
per_pair_rates = {}
if result.swap_history:
    from collections import Counter

    _pair_proposed = Counter()
    _pair_accepted = Counter()
    for entry in result.swap_history:
        pair = entry["pair"]
        _pair_proposed[pair] += 1
        if entry["accepted"]:
            _pair_accepted[pair] += 1
    for pair in sorted(_pair_proposed.keys()):
        i, j = pair
        key = f"{BETAS[i]}↔{BETAS[j]}"
        per_pair_rates[key] = _pair_accepted[pair] / _pair_proposed[pair]

torch.save(
    {
        "chains": chains_payload,
        "params": run_params,
        "swap_diagnostics": {
            "n_swaps_proposed": result.n_swaps_proposed,
            "n_swaps_accepted": result.n_swaps_accepted,
            "swap_acceptance_rate": float(result.swap_acceptance_rate()),
            "per_pair_rates": per_pair_rates,
        },
        "swap_history": result.swap_history if result.swap_history else [],
        "mle_param": mle_param.cpu(),
        # H3-1 seed: per-chain adapted mass matrices and step sizes
        "per_chain_adapted_M": per_chain_M,
        "per_chain_adapted_step_size": per_chain_eps,
    },
    tempered_path,
)

with open(RESULTS_DIR / "params.json", "w") as f:
    json.dump(run_params, f, indent=2)

print(f"✓ Saved H3 initial run to {tempered_path}")
print(f"  {N_CHAINS} chains × {N_SAMPLES} samples × d={mle_param.shape[0]}")
print(f"  Per-chain adapted M / ε captured for H3-1 seed")


## H3-1: Iterative Adaptation Refinement (mirrors H1-1)

**Hypothesis H3-1**: Starting from H3's per-chain adapted mass matrices,
run repeated rounds of (warmup → production → use new per-chain M's).
Each round saves a checkpoint so the analysis notebook can recompute
ACF/ESS per round without rerunning sampling.

Each round:
1. Re-warmup PT using the previous round's per-chain M / ε as seed
   (each chain continues adapting its own M).
2. Production: PT with fixed per-chain M.
3. Save samples + per-chain M + swap diagnostics → `iterative/round_NN.pt`.


In [ ]:
# ── H3-1 setup ──
import json as _json

if RUN_ITERATIVE:
    ITER_DIR.mkdir(parents=True, exist_ok=True)

    # Seed: per-chain adapted M / ε from the initial H3 run
    iter_per_chain_M = list(per_chain_M)
    iter_per_chain_eps = list(per_chain_eps)

    start_round = 0

    if RESUME:
        existing = sorted(ITER_DIR.glob("round_*.pt"))
        if existing:
            last_path = existing[-1]
            last = torch.load(last_path, weights_only=False)
            start_round = int(last["round"])
            iter_per_chain_M = last["per_chain_mass_matrix"]
            iter_per_chain_eps = last["per_chain_step_size"]
            print(f"Resuming from {last_path.name} (round {start_round})")

    print(
        f"H3-1: {N_ROUNDS} rounds × ({N_WARMUP_PER_ROUND} warmup + "
        f"{N_SAMPLES_PER_ROUND} production)"
    )
    print(f"Starting at round {start_round + 1}")


In [ ]:
# ── H3-1 iterative loop ──
if RUN_ITERATIVE:
    round_results = []

    summary_path = ITER_DIR / "round_summary_raw.json"
    if RESUME and summary_path.exists():
        with open(summary_path) as f:
            round_results = _json.load(f)

    for rnd in range(start_round, N_ROUNDS):
        print(f"\n{'=' * 70}")
        print(f"ROUND {rnd + 1}/{N_ROUNDS}")
        print(f"{'=' * 70}")

        t_round = time.time()

        # Warmup: each chain re-adapts its own M, seeded from previous round's
        # per-chain M / ε via the new per_chain_mass_matrix / per_chain_step_size API.
        warmup_r = sampler.sample(
            config=NUTS(
                n_warmup=N_WARMUP_PER_ROUND,
                step_size=iter_per_chain_eps[0],  # fallback only
                max_tree_depth=MAX_TREE_DEPTH,
                target_accept=TARGET_ACCEPT,
                adapt_mass_matrix=True,
            ),
            n_samples=1,
            n_chains=N_CHAINS,
            init_strategy=Perturb(scale=INIT_PERTURB_SCALE),
            swap_every=SWAP_EVERY,
            betas=BETAS,
            per_chain_mass_matrix=iter_per_chain_M,
            per_chain_step_size=iter_per_chain_eps,
        )

        # Capture newly adapted per-chain M / ε
        new_per_chain_M = []
        new_per_chain_eps = []
        for ci, ch in enumerate(warmup_r.chains):
            d_w = ch.diagnostics
            M_w = d_w.get("adapted_mass_matrix")
            eps_w = d_w.get(
                "adapted_step_size", d_w.get("step_size", iter_per_chain_eps[ci])
            )
            new_per_chain_M.append(M_w.cpu() if M_w is not None else iter_per_chain_M[ci])
            new_per_chain_eps.append(float(eps_w))

        iter_per_chain_M = new_per_chain_M
        iter_per_chain_eps = new_per_chain_eps

        # Production: PT with fixed per-chain M (no further adaptation).
        prod_r = sampler.sample(
            config=NUTS(
                n_warmup=0,
                step_size=iter_per_chain_eps[0],  # fallback only
                max_tree_depth=MAX_TREE_DEPTH,
                target_accept=TARGET_ACCEPT,
                adapt_mass_matrix=False,
            ),
            n_samples=N_SAMPLES_PER_ROUND,
            n_chains=N_CHAINS,
            init_strategy=Perturb(scale=INIT_PERTURB_SCALE),
            swap_every=SWAP_EVERY,
            betas=BETAS,
            per_chain_mass_matrix=iter_per_chain_M,
            per_chain_step_size=iter_per_chain_eps,
        )

        elapsed_round = time.time() - t_round

        # Cold-chain summary (round-level diagnostics mirror H1-1's structure)
        cold_ch = prod_r.chains[0]
        cold_samps = torch.stack(cold_ch.parameters).cpu().float()
        cold_diag = cold_ch.diagnostics
        M0_cpu = (
            iter_per_chain_M[0].detach().cpu().float()
            if iter_per_chain_M[0] is not None
            else None
        )
        if M0_cpu is not None:
            sv0 = torch.linalg.svdvals(M0_cpu)
            cond_M0 = float(sv0[0] / sv0[-1]) if sv0[-1] > 0 else float("inf")
        else:
            cond_M0 = float("nan")

        # Per-pair swap rates this round
        pair_rates_round = {}
        if prod_r.swap_history:
            from collections import Counter as _Counter

            pp = _Counter()
            pa = _Counter()
            for e in prod_r.swap_history:
                pp[e["pair"]] += 1
                if e["accepted"]:
                    pa[e["pair"]] += 1
            for pair in sorted(pp.keys()):
                i, j = pair
                pair_rates_round[f"{BETAS[i]}↔{BETAS[j]}"] = pa[pair] / pp[pair]

        round_info = {
            "round": rnd + 1,
            "step_size_cold": float(iter_per_chain_eps[0]),
            "cond_M_cold": cond_M0,
            "acceptance_rate_cold": float(cold_ch.acceptance_rate),
            "n_divergences_cold": int(cold_diag.get("n_divergences", 0)),
            "mean_tree_depth_cold": float(cold_diag.get("mean_tree_depth", 0)),
            "n_samples": int(cold_samps.shape[0]),
            "swap_acceptance_rate": float(prod_r.swap_acceptance_rate()),
            "n_swaps_proposed": int(prod_r.n_swaps_proposed),
            "n_swaps_accepted": int(prod_r.n_swaps_accepted),
            "per_pair_swap_rates": pair_rates_round,
            "elapsed_seconds": elapsed_round,
            "per_chain_step_size": list(iter_per_chain_eps),
        }
        round_results.append(round_info)

        # Per-round checkpoint (analysis recomputes ACF/ESS from samples + hessian.pt).
        # Store all chains so analysis can also inspect non-cold chains if needed.
        all_chains_payload = [
            {
                "parameters": torch.stack(ch.parameters).cpu(),
                "acceptance_rate": float(ch.acceptance_rate),
                "diagnostics": ch.diagnostics,
                "beta": BETAS[ci],
            }
            for ci, ch in enumerate(prod_r.chains)
        ]

        torch.save(
            {
                "round": rnd + 1,
                "per_chain_mass_matrix": [
                    (m.cpu() if m is not None else None) for m in iter_per_chain_M
                ],
                "per_chain_step_size": list(iter_per_chain_eps),
                "samples": cold_samps,  # cold-chain (β=1) samples for fast analysis
                "chains": all_chains_payload,
                "swap_history": prod_r.swap_history if prod_r.swap_history else [],
                "diagnostics": round_info,
                "betas": BETAS,
            },
            ITER_DIR / f"round_{rnd + 1:02d}.pt",
        )

        with open(summary_path, "w") as f:
            _json.dump(round_results, f, indent=2)

        print(
            f"  ε_cold={iter_per_chain_eps[0]:.4e}  cond(M_cold)={cond_M0:.2e}  "
            f"accept_cold={round_info['acceptance_rate_cold']:.3f}  "
            f"swap={round_info['swap_acceptance_rate']:.3f}  "
            f"divs={round_info['n_divergences_cold']}"
        )
        print(f"  ✓ Saved round_{rnd + 1:02d}.pt + round_summary_raw.json")

    print(f"\n{'=' * 70}")
    print(f"H3-1 complete: {len(round_results)} rounds total")
    print(f"Checkpoints in {ITER_DIR}")
